In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd 

from scripts.analytics import *
from scripts.algorithms import *


In [2]:
name = "productivity"
namex = "Productivity "
d = 11

###knn
graphs = np.load("../graphs/"+name+"_knn_random.npy", allow_pickle=True)
###threshold
graphsz = np.load("../graphs/"+name+"_thresh_random.npy", allow_pickle=True)


# Compute bruteforce algorithm results

## Nearest neighbor

In [3]:
### read the random, greedy results
df_greedy1 = pd.read_csv("./sm_results/"+name+"_knn_summary.csv")
df_greedy1.head()

,K,F(Sg+ U Sg-)*,(Sg+ U Sg-)*,F(Sr+ U Sr-)*,(Sr+ U Sr-)*,F(Sg),Sg,F(Sr),Sr,F(Sgfull),...,F(Sg)/F(Sr),F(Sg)/F(Sgfull),F(Sr)/F(Srfull),budgetGreedyTime,budgetRdmTime,kmax,n,m,dataset,graphid
0,1,77.0,set(),77.0,{7},77.0,set(),77.0,{11},77.0,...,1.0,1.0,1.0,0.006330,0.000230,1,112,32,productivity (11),0
1,2,77.0,set(),77.0,"{34, 7}",77.0,set(),77.0,"{11, 4}",77.0,...,1.0,1.0,1.0,0.006751,0.000230,1,112,32,productivity (11),0
2,3,77.0,set(),77.0,"{34, 18, 7}",77.0,set(),77.0,"{27, 11, 4}",77.0,...,1.0,1.0,1.0,0.007343,0.000326,1,112,32,productivity (11),0
3,4,77.0,set(),77.0,"{11, 34, 18, 7}",77.0,set(),77.0,"{27, 11, 4, 22}",77.0,...,1.0,1.0,1.0,0.007161,0.000239,1,112,32,productivity (11),0
4,5,77.0,set(),77.0,"{34, 18, 7, 11, 29}",77.0,set(),77.0,"{4, 11, 21, 22, 27}",77.0,...,1.0,1.0,1.0,0.006854,0.000269,1,112,32,productivity (11),0


In [4]:
greedy_bruteforce_algosdf = pd.DataFrame()
all_bruteforce = pd.DataFrame()

for lx in range(len(graphs)):

    r_edges_random  = graphs[lx]["edges"]
    r_labels_random = graphs[lx]["labels"]
    
    optls = budget_opts(edges=r_edges_random, labels=r_labels_random, budget=5)
    df_optls = pd.DataFrame(optls)
    df_optls["graphidx"] = [lx]*len(df_optls)
    all_bruteforce = pd.concat([all_bruteforce, df_optls], ignore_index=True)


    df_greedy11 = df_greedy1[df_greedy1["graphid"]==lx][['K', 'Sg', 'F(Sg)', 'Sr', 'F(Sr)', '(Sg+ U Sg-)*',
                                                         'F(Sg+ U Sg-)*', '(Sr+ U Sr-)*', 'F(Sr+ U Sr-)*', 'kmax']]

    merged = pd.merge(df_greedy11, df_optls, left_on="K", right_on="b", how="inner").drop(columns=["b"])
    merged["F(Sg)/F(S*)"] = merged["F(Sg)"] / merged["F(S*)"]
    merged["F(Sr)/F(S*)"] = merged["F(Sr)"] / merged["F(S*)"]
    merged["F(Sg+ U Sg-)* / F(S*)"] = merged["F(Sg+ U Sg-)*"] / merged["F(S*)"]
    merged["F(Sr+ U Sr-)* / F(S*)"] = merged["F(Sr+ U Sr-)*"] / merged["F(S*)"]

    summary_dfv = merged[[
        "K", "Sg", "F(Sg)", "Sr", "F(Sr)", "(Sg+ U Sg-)*", "F(Sg+ U Sg-)*",
        "(Sr+ U Sr-)*", "F(Sr+ U Sr-)*", "F(S*)", "S*",
        "F(Sg)/F(S*)", "F(Sr)/F(S*)", "F(Sg+ U Sg-)* / F(S*)", "F(Sr+ U Sr-)* / F(S*)",
        "BFtime", "kmax"
    ]]

    summary_df = summary_dfv.copy()
    summary_df["n"] = [graphs[lx]['n']]*len(summary_df)
    summary_df["m"] = [graphs[lx]['m']]*len(summary_df)
    summary_df["dataset"] = [name+" ("+ str(d) + ")"]*len(summary_df)
    summary_df["graphid"] = [lx]*len(summary_df)

    
    greedy_bruteforce_algosdf = pd.concat([greedy_bruteforce_algosdf, summary_df], ignore_index=True)



## Thresholding

In [5]:
df_greedy1z = pd.read_csv("./sm_results/"+name+"_thresh_summary.csv")
df_greedy1z.head()

,K,F(Sg+ U Sg-)*,(Sg+ U Sg-)*,F(Sr+ U Sr-)*,(Sr+ U Sr-)*,F(Sg),Sg,F(Sr),Sr,F(Sgfull),...,F(Sg)/F(Sr),F(Sg)/F(Sgfull),F(Sr)/F(Srfull),budgetGreedyTime,budgetRdmTime,r,n,m,dataset,graphid
0,1,96.845464,{2},88.545778,{9},96.845464,{2},88.194736,{40},104.0,...,1.098087,0.931206,0.848026,0.071540,0.001525,4.0,112,50,productivity (11),0
1,2,103.500000,"{2, 31}",88.760239,"{9, 29}",103.500000,"{2, 31}",88.385736,"{40, 7}",104.0,...,1.171003,0.995192,0.849863,0.143423,0.000944,4.0,112,50,productivity (11),0
2,3,104.000000,"{16, 2, 31}",88.951239,"{9, 29, 7}",104.000000,"{16, 2, 31}",88.830180,"{40, 1, 7}",104.0,...,1.170773,1.000000,0.854136,0.206981,0.001401,4.0,112,50,productivity (11),0
3,4,104.000000,"{16, 2, 31}",102.560160,"{9, 20, 1, 22}",104.000000,"{16, 2, 31}",89.115895,"{40, 1, 47, 7}",104.0,...,1.167020,1.000000,0.856884,0.252051,0.000898,4.0,112,50,productivity (11),0
4,5,104.000000,"{16, 2, 31}",102.795455,"{1, 19, 20, 22, 9}",104.000000,"{16, 2, 31}",89.115895,"{1, 7, 40, 47, 17}",104.0,...,1.167020,1.000000,0.856884,0.324906,0.001704,4.0,112,50,productivity (11),0


In [6]:
greedy_bruteforce_algosdfz = pd.DataFrame()
all_bruteforcez = pd.DataFrame()

for lz in range(len(graphsz)):

    r_edges_randomz  = graphsz[lz]["edges"]
    r_labels_randomz = graphsz[lz]["labels"]
    
    optlsz = budget_opts(edges=r_edges_randomz, labels=r_labels_randomz, budget=5)
    
    df_optlsz = pd.DataFrame(optlsz)
    df_optlsz["graphidx"] = [lz]*len(df_optlsz)
    all_bruteforcez = pd.concat([all_bruteforcez, df_optlsz], ignore_index=True)


    df_greedy11z = df_greedy1z[df_greedy1z["graphid"]==lz][['K', 'Sg', 'F(Sg)', 'Sr', 'F(Sr)', '(Sg+ U Sg-)*',
                                                         'F(Sg+ U Sg-)*', '(Sr+ U Sr-)*', 'F(Sr+ U Sr-)*', 'r']]

    mergedz = pd.merge(df_greedy11z, df_optlsz, left_on="K", right_on="b", how="inner").drop(columns=["b"])
    mergedz["F(Sg)/F(S*)"] = mergedz["F(Sg)"] / mergedz["F(S*)"]
    mergedz["F(Sr)/F(S*)"] = mergedz["F(Sr)"] / mergedz["F(S*)"]
    mergedz["F(Sg+ U Sg-)* / F(S*)"] = mergedz["F(Sg+ U Sg-)*"] / mergedz["F(S*)"]
    mergedz["F(Sr+ U Sr-)* / F(S*)"] = mergedz["F(Sr+ U Sr-)*"] / mergedz["F(S*)"]

    summary_dfzv = mergedz[[
        "K", "Sg", "F(Sg)", "Sr", "F(Sr)", "(Sg+ U Sg-)*", "F(Sg+ U Sg-)*",
        "(Sr+ U Sr-)*", "F(Sr+ U Sr-)*", "F(S*)", "S*",
        "F(Sg)/F(S*)", "F(Sr)/F(S*)", "F(Sg+ U Sg-)* / F(S*)", "F(Sr+ U Sr-)* / F(S*)",
        "BFtime", "r"
    ]]

    summary_dfz = summary_dfzv.copy()
    summary_dfz["n"] = [graphsz[lz]['n']]*len(summary_dfz)
    summary_dfz["m"] = [graphsz[lz]['m']]*len(summary_dfz)
    summary_dfz["dataset"] = [name+" ("+ str(d) + ")"]*len(summary_dfz)
    summary_dfz["graphid"] = [lz]*len(summary_dfz)

    
    greedy_bruteforce_algosdfz = pd.concat([greedy_bruteforce_algosdfz, summary_dfz], ignore_index=True)



# Save Results 

In [7]:
greedy_bruteforce_algosdf.head()

,K,Sg,F(Sg),Sr,F(Sr),(Sg+ U Sg-)*,F(Sg+ U Sg-)*,(Sr+ U Sr-)*,F(Sr+ U Sr-)*,F(S*),...,F(Sg)/F(S*),F(Sr)/F(S*),F(Sg+ U Sg-)* / F(S*),F(Sr+ U Sr-)* / F(S*),BFtime,kmax,n,m,dataset,graphid
0,1,set(),77.0,{11},77.0,set(),77.0,{7},77.0,77.0,...,1.0,1.0,1.0,1.0,0.009695,1,112,32,productivity (11),0
1,2,set(),77.0,"{11, 4}",77.0,set(),77.0,"{34, 7}",77.0,77.0,...,1.0,1.0,1.0,1.0,0.137197,1,112,32,productivity (11),0
2,3,set(),77.0,"{27, 11, 4}",77.0,set(),77.0,"{34, 18, 7}",77.0,77.0,...,1.0,1.0,1.0,1.0,1.402950,1,112,32,productivity (11),0
3,4,set(),77.0,"{27, 11, 4, 22}",77.0,set(),77.0,"{11, 34, 18, 7}",77.0,77.0,...,1.0,1.0,1.0,1.0,9.622657,1,112,32,productivity (11),0
4,5,set(),77.0,"{4, 11, 21, 22, 27}",77.0,set(),77.0,"{34, 18, 7, 11, 29}",77.0,77.0,...,1.0,1.0,1.0,1.0,54.050180,1,112,32,productivity (11),0


In [8]:
all_bruteforce.head()

,b,F(S*),S*,BFtime,graphidx
0,0,77.0,{},0.000560,0
1,1,77.0,{},0.009695,0
2,2,77.0,{},0.137197,0
3,3,77.0,{},1.402950,0
4,4,77.0,{},9.622657,0


In [9]:
greedy_bruteforce_algosdfz.head()

,K,Sg,F(Sg),Sr,F(Sr),(Sg+ U Sg-)*,F(Sg+ U Sg-)*,(Sr+ U Sr-)*,F(Sr+ U Sr-)*,F(S*),...,F(Sg)/F(S*),F(Sr)/F(S*),F(Sg+ U Sg-)* / F(S*),F(Sr+ U Sr-)* / F(S*),BFtime,r,n,m,dataset,graphid
0,1,{2},96.845464,{40},88.194736,{2},96.845464,{9},88.545778,96.845464,...,1.0,0.910675,1.0,0.914300,0.056395,4.0,112,50,productivity (11),0
1,2,"{2, 31}",103.500000,"{40, 7}",88.385736,"{2, 31}",103.500000,"{9, 29}",88.760239,103.500000,...,1.0,0.853968,1.0,0.857587,1.252056,4.0,112,50,productivity (11),0
2,3,"{16, 2, 31}",104.000000,"{40, 1, 7}",88.830180,"{16, 2, 31}",104.000000,"{9, 29, 7}",88.951239,104.000000,...,1.0,0.854136,1.0,0.855300,20.290961,4.0,112,50,productivity (11),0
3,4,"{16, 2, 31}",104.000000,"{40, 1, 47, 7}",89.115895,"{16, 2, 31}",104.000000,"{9, 20, 1, 22}",102.560160,104.000000,...,1.0,0.856884,1.0,0.986155,250.146892,4.0,112,50,productivity (11),0
4,5,"{16, 2, 31}",104.000000,"{1, 7, 40, 47, 17}",89.115895,"{16, 2, 31}",104.000000,"{1, 19, 20, 22, 9}",102.795455,104.000000,...,1.0,0.856884,1.0,0.988418,2271.425931,4.0,112,50,productivity (11),0


In [10]:
all_bruteforcez.head()

,b,F(S*),S*,BFtime,graphidx
0,0,80.725188,{},0.002735,0
1,1,96.845464,{2},0.056395,0
2,2,103.500000,"{2, 31}",1.252056,0
3,3,104.000000,"{16, 1, 2}",20.290961,0
4,4,104.000000,"{16, 1, 2}",250.146892,0


In [11]:
greedy_bruteforce_algosdf.to_csv("./sm_results/"+name+"_knn_greedy_rdm_bruteforce.csv", index=False)
all_bruteforce.to_csv("./sm_results/"+name+"_knn_allbruteforce.csv", index=False)

greedy_bruteforce_algosdfz.to_csv("./sm_results/"+name+"_thresh_greedy_rdm_bruteforce.csv", index=False)
all_bruteforcez.to_csv("./sm_results/"+name+"_thresh_allbruteforce.csv", index=False)
